# Математическая статистика для анализа больших данных
## Семинар 6
### Множественная проверка гипотез

Выполненную работу нужно отправить телеграм-боту  `@stats_bd_bot`
* Дедлайн см. в телеграм-боте. После дедлайна работы не принимаются кроме случаев наличия уважительной причины.
* По практическим задачам прислать нужно ноутбук в формате `ipynb`.
* Решения, размещенные на каких-либо интернет-ресурсах не принимаются. Кроме того, публикация решения в открытом доступе может быть приравнена к предоставлении возможности списать.
* Не забывайте делать пояснения и выводы
* Решение теоретических задач можете записывать в markdown с помощью TeX, присылать в виде фотографий или вшивать в ноутбук через Insert Image (убеждайтесь, что картинка сохраняется при перемещении ноутбука в другое место)

In [ ]:
# Bot check

# HW_ID: sbd_sem6
# Бот проверит этот ID и предупредит, если случайно сдать что-то не то.

# Status: not final
# Перед отправкой в финальном решении удали "not" в строчке выше.
# Так бот проверит, что ты отправляешь финальную версию, а не промежуточную.
# Никакие значения в этой ячейке не влияют на факт сдачи работы.

In [14]:
import numpy as np
import scipy.stats as sps
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as wg
from ipywidgets import interact, Layout
from IPython.display import display
import pandas as pd
from statsmodels.stats.multitest import multipletests

sns.set(font_scale=1.5)

Были проведены эксперименты для оценки эффективности нескольких препаратов для снижения послеоперационной тошноты. Результаты экспериментов приведены в таблице ниже. При проведении эксперимента пациенты делились на группы случайным образом.




|| Количество пациентов   | Количество случаев возникновения тошноты |
|-----------| ----------- | -------|
|Плацебо|80|45|
|Хлорпромазин|75|26|
|Дименгидринат|85|52|
|Пентобарбитал (100 мг)|67|35|
|Пентобарбитал (150 мг)|85|37|




Проведите сравнение каждого препарата по эффективности по отношению к плацебо. Какие ответы можно получить для методов, контролирующих FWER и FDR? В каждом случае приведите подправленные p-value.

*Замечание.* Используйте [`multipletests`](https://www.statsmodels.org/dev/generated/statsmodels.stats.multitest.multipletests.html) из библиотеки `statsmodels`.

Для каждой пары плацебо / препарат нас имеются две бернуллиевские выборки: $$X = (X_1, \ldots, X_n) \sim Bern(\theta_1) \\ Y = (Y_1, \ldots, Y_m) \sim Bern(\theta_2)$$
Первая из которых будет соответствовать какому-то препарату (индикатор того, что тошнота **отсутствует**), а вторая &mdash; плацебо.

Проверяется $\mathsf{H}_0 \colon \theta_1 = \theta_2 \ vs. \ \mathsf{H}_1 \colon \theta_1 > \theta_2$.

Будем использовать критерий Вальда:
$$
W(X, Y) = \frac{\overline{X} -\overline{Y}}{\widehat{\sigma}} \xrightarrow[]{d_0} \mathcal{N}(0, 1) \\
\widehat{\sigma} = \sqrt{\frac{\overline{X}(1 - \overline{X})}{n} + \frac{\overline{Y}(1 - \overline{Y})}{m}}
$$

Выпишите критерий:

Если $W > z_{1 - α}$, то отвергаем $H_0$ в пользу $H_1$.
Или, эквивалентно, если $p-value < \alpha$, где $p-value$ вычисляется по статистике $W$.

Формула подсчета p-value: `p-value(t) = 1 - sps.norm.cdf(W)
`

Для удобства выделим подсчет статистики Вальда в функцию.

In [1]:
def wald_stats(x_mean, n, y_mean, m):
    '''
    Подсчет значения статистики Вальда.

    :param x_mean: среднее значение по выборке X (препарат)
    :param n: размер выборки X
    :param y_mean: среднее значение по выборке Y (плацебо)
    :param m: размер выборки Y

    :return: посчитанное значение статистики Вальда
    '''
    numerator = x_mean - y_mean
    sigma_hat = np.sqrt(
        (x_mean * (1 - x_mean) / n) +
        (y_mean * (1 - y_mean) / m)
    )
    W = numerator / sigma_hat
    return W

Информация о плацебо.

In [2]:
placebo_m = 80  # размер выборки для плацебо
placebo_sum = 45  # количество случаев тошноты
# среднее по тем, у кого не было тошноты
placebo_mean = (placebo_m - placebo_sum) / placebo_m

Анологично информация по лекарствам

In [3]:
# Хлорпромазин
chlorp_n = 75
chlorp_sum = 26
chlorp_mean = (chlorp_n - chlorp_sum) / chlorp_n

# Дименгидринат
dimen_n = 85
dimen_sum = 52
dimen_mean = (dimen_n - dimen_sum) / dimen_n

# Пентобарбитал (100 мг)
pent100_n = 67
pent100_sum = 35
pent100_mean = (pent100_n - pent100_sum) / pent100_n

# Пентобарбитал (150 мг)
pent150_n = 85
pent150_sum = 37
pent150_mean = (pent150_n - pent150_sum) / pent150_n

Посмотрим на результаты проверки гипотезы без МПГ

In [4]:
alpha=0.05

In [12]:
pvals = []

names = ["Хлорпромазин", "Дименгидринат", "Пентобарбитал (100 мг)", "Пентобарбитал (150 мг)"]
means = [chlorp_mean, dimen_mean, pent100_mean, pent150_mean]
nums = [chlorp_n, dimen_n, pent100_n, pent150_n]
for medicine_mean, medicine_n, medicine_name in zip(means, nums, names):
    wald = wald_stats(medicine_mean, medicine_n, placebo_mean, placebo_m)
    p_value = 1 - sps.norm.cdf(wald)
    pvals.append(p_value)

    print(f'Лекарство {medicine_name}: W = {wald:.2f}, p-value = {p_value:.3f}, отвергаем при p-value < 0.05')

Лекарство Хлорпромазин: W = 2.76, p-value = 0.003, отвергаем при p-value < 0.05
Лекарство Дименгидринат: W = -0.64, p-value = 0.740, отвергаем при p-value < 0.05
Лекарство Пентобарбитал (100 мг): W = 0.49, p-value = 0.313, отвергаем при p-value < 0.05
Лекарство Пентобарбитал (150 мг): W = 1.65, p-value = 0.050, отвергаем при p-value < 0.05


Посмотрим на результаты проверки гипотезы c использованием МПГ

In [16]:
res_holm, pvals_holm= multipletests(pvals, alpha=0.05, method='holm')[:2] # Метод Холма
res_by, pvals_by= multipletests(pvals, alpha=0.05, method='fdr_by')[:2] # Метод Бенджамини-Иекутиели

for holm, by, medicine_name in zip(res_holm, res_by, names):
    print(f'Лекарство {medicine_name}: отвергаем (Холм) {holm}, отвергаем (Бенджамини-Иекутиели) {by}')

Лекарство Хлорпромазин: отвергаем (Холм) True, отвергаем (Бенджамини-Иекутиели) True
Лекарство Дименгидринат: отвергаем (Холм) False, отвергаем (Бенджамини-Иекутиели) False
Лекарство Пентобарбитал (100 мг): отвергаем (Холм) False, отвергаем (Бенджамини-Иекутиели) False
Лекарство Пентобарбитал (150 мг): отвергаем (Холм) False, отвергаем (Бенджамини-Иекутиели) False


Соберем значения pvalue в общую табличку.

In [17]:
pd.DataFrame([pvals, pvals_holm, pvals_by],
             columns=names,
             index=['pvalue до коррекции', 'метод Холма', 'метод Бенджамини-Иекутиели'])

,Хлорпромазин,Дименгидринат,Пентобарбитал (100 мг),Пентобарбитал (150 мг)
pvalue до коррекции,0.002852,0.739884,0.313332,0.049820
метод Холма,0.011407,0.739884,0.626664,0.149459
метод Бенджамини-Иекутиели,0.023764,1.000000,0.870367,0.207582


Как изменились результаты проверки гипотезы? Какие лекарства можно считать эффективными? Сделайте выводы.

---
**Вывод:**
После проведения коррекции на МПГ, результаты проверки изменились. Без применения МПГ препараты хлорпромазин и пентобарбитал (150 мг) показали статистически значимую эффективность по сравнению с плацебо при уровне значимости 0.05. Однако после применения методов Холма (контролирующего FWER) и Бенджамини-Иекутиели (контролирующего FDR) для корректировки p-value в связи с МПГ, только хлорпромазин сохранил статистическую значимость. Это означает, что с учетом МПГ только хлорпромазин можно считать эффективным в снижении послеоперационной тошноты по сравнению с плацебо. Остальные препараты не продемонстрировали статистически значимого эффекта после корректировки p-value с учетом МПГ, поэтому их эффективность не подтверждается данными исследования. Таким образом, корректировка на МПГ существенно повлияла на выводы о эффективности препаратов.

---